# MNIST Handwritten Digit Classifier

This notebook downloads MNIST, prepares training and test datasets, trains a simple neural-network classifier, and reports test-set performance.

**Run in Google Colab:** choose **Runtime → Run all**. A GPU is optional; the model is intentionally small and also runs on CPU.

**Dataset:** 60,000 training images and 10,000 test images, each a 28×28 grayscale image labeled as a digit from 0 to 9.

## 1. Import libraries and check the runtime

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

print("TensorFlow version:", tf.__version__)
print("Available GPU devices:", tf.config.list_physical_devices("GPU"))

## 2. Download MNIST and create train/test datasets

In [ ]:
# Keras downloads MNIST automatically on the first run and caches it afterward.
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

print("Training images:", x_train.shape)
print("Training labels:", y_train.shape)
print("Test images:", x_test.shape)
print("Test labels:", y_test.shape)

# Convert pixel values from [0, 255] to float32 [0, 1].
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# Keep labels as integer class IDs (0 through 9).
y_train = y_train.astype("int64")
y_test = y_test.astype("int64")

assert x_train.shape == (60000, 28, 28)
assert x_test.shape == (10000, 28, 28)
assert np.isfinite(x_train).all() and np.isfinite(x_test).all()
assert x_train.min() >= 0 and x_train.max() <= 1

print("Dataset preparation checks passed.")

## 3. Inspect a few training examples

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(x_train[i], cmap="gray")
    ax.set_title(f"Label: {y_train[i]}")
    ax.axis("off")
plt.tight_layout()
plt.show()

## 4. Build and train a simple classifier

The model flattens each 28×28 image, uses one hidden dense layer, and outputs 10 class probabilities. The final layer uses softmax, and the loss is sparse categorical cross-entropy.

In [ ]:
tf.keras.utils.set_random_seed(42)

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(28, 28)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(10, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

history = model.fit(
    x_train,
    y_train,
    epochs=5,
    batch_size=128,
    validation_split=0.1,
    verbose=1
)

## 5. Evaluate on the held-out test dataset

In [ ]:
test_loss, test_accuracy = model.evaluate(x_test, y_test, verbose=0)

print(f"Test loss:     {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4%}")
print(f"Test error:    {1.0 - test_accuracy:.4%}")

## 6. Report classification errors

In [ ]:
# Predict the most likely digit for each test image.
probabilities = model.predict(x_test, batch_size=256, verbose=0)
y_pred = np.argmax(probabilities, axis=1)

# A prediction is incorrect when its predicted class differs from the true label.
wrong_indices = np.flatnonzero(y_pred != y_test)
error_count = len(wrong_indices)
error_rate = error_count / len(y_test)

print(f"Incorrect predictions: {error_count:,} / {len(y_test):,}")
print(f"Misclassification rate: {error_rate:.4%}")

# Show up to 12 mistakes for inspection.
n_show = min(12, error_count)
if n_show:
    fig, axes = plt.subplots(3, 4, figsize=(10, 7))
    for ax, idx in zip(axes.flat, wrong_indices[:n_show]):
        ax.imshow(x_test[idx], cmap="gray")
        ax.set_title(
            f"True: {y_test[idx]} | Pred: {y_pred[idx]}\n"
            f"Confidence: {probabilities[idx, y_pred[idx]]:.1%}"
        )
        ax.axis("off")
    for ax in axes.flat[n_show:]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("No test errors to display.")

## 7. Confusion matrix and per-class report

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

cm = confusion_matrix(y_test, y_pred, labels=np.arange(10))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=np.arange(10))
fig, ax = plt.subplots(figsize=(8, 8))
disp.plot(ax=ax, cmap="Blues", colorbar=False, values_format="d")
plt.title("MNIST test-set confusion matrix")
plt.show()

print(classification_report(y_test, y_pred, digits=4, zero_division=0))

## 8. Save the trained model (optional)

This saves the model in the Colab session. To keep it after the session ends, download the file or save it to Google Drive.

In [ ]:
model.save("mnist_classifier.keras")
print("Saved model to mnist_classifier.keras")

## Results summary

The cells above report:
- Test loss and accuracy
- Test error rate (1 − accuracy)
- Number and examples of incorrect predictions
- Confusion matrix
- Precision, recall, and F1-score for each digit class

**Interpretation:** test error is the fraction of the 10,000 held-out test images classified incorrectly. The displayed results are produced when you run the notebook; they are not hard-coded.